# Bass lagu penuh — konteks 30 detik

Uji pengguna pada notebook 05: input chunk 2 tetap glitch saat diulang dalam
proses baru, sedangkan satu generasi 30 detik pertama bersih. Notebook ini memakai
konteks 30 detik sepanjang lagu, dengan overlap nominal 3 detik. Untuk 180 detik
ada 7 chunk, termasuk potongan akhir yang disesuaikan agar tetap punya konteks penuh.

**Taruh rekaman di `data/full_song/`**, lalu Restart Kernel → Run All.
Runtime, bobot Base, seed 0, MPS dan CPU offload tetap sama. Tidak ada instalasi
atau unduhan baru. Model dimuat sekali untuk chunk yang belum selesai.

Run disimpan terpisah dari percobaan 10 detik; cache yang glitch tidak dipakai.
Run dengan sumber dan pengaturan sama dapat dilanjutkan. Satu uji 30 detik bersih
belum membuktikan bahwa semua bagian lagu bebas glitch. Tiap chunk tetap digenerasi
mandiri: karakter/pola bass bisa berubah pada sambungan. Arsitektur/model tidak
berubah dan belum ada training baru. Ini langkah evaluasi lagu penuh, bukan bukti
kemampuan on-device iPhone.


In [ ]:
from pathlib import Path
import hashlib, json, sys
from IPython.display import Audio, display
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs/mac-full-song.json").is_file(), "Buka dari checkout StemsGenerating-Models"
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "tools/stem_eval"))
from mac_workflow import choose_source, check_model_runtime
from setup_runtime import runtime_python
from stream_process import run_logged
from run_song import song_identity
from system_tools import require_audio_tools
SONG_DATA = ROOT / "data/full_song"
SONG_DATA.mkdir(parents=True, exist_ok=True)


## 1. Pilih rekaman sendiri

`SOURCE=None` memilih satu rekaman di `data/full_song/`. Folder terpisah ini
menghindari pemilihan input.wav lama milik notebook 02. Untuk memilih path lain,
isi SOURCE dengan Path yang sesuai. DRY_RUN=True hanya menyiapkan dan merencanakan
potongan, tanpa model; default False menjalankan generasi nyata.


In [ ]:
SOURCE = None
DRY_RUN = False
require_audio_tools()
source = choose_source(SONG_DATA, SOURCE)
PROFILE_PATH = ROOT / "configs/mac-full-song.json"
PROFILE = json.loads(PROFILE_PATH.read_text())
CHUNK_SECONDS = 30
OVERLAP_SECONDS = 3
identity = song_identity(source, PROFILE, ROOT / "configs/assets-base.json", CHUNK_SECONDS, OVERLAP_SECONDS)
run_key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:20]
RUN = ROOT / "runs" / ("full_song30_" + run_key)
print("Sumber:", source)
print("Profil:", json.dumps(PROFILE, indent=2))
print("Folder run/resume:", RUN)


## 2. Persiapkan seluruh lagu dan periksa timeline

Rencana mencatat durasi penuh, jumlah chunk, hash audio, dan posisi berdasarkan
frame sampel. File sumber tidak diubah. Output dan input yang berubah checksum
tidak dipakai diam-diam untuk resume. Sumber yang lebih pendek dari 30 detik
memiliki satu chunk dengan padding; final output dipotong kembali ke panjang asli.


In [ ]:
command = [str(ROOT / "scripts/run_song.py"), "--source", str(source),
           "--out", str(RUN), "--profile", str(PROFILE_PATH),
           "--chunk-seconds", str(CHUNK_SECONDS), "--overlap-seconds", str(OVERLAP_SECONDS)]
run_logged([sys.executable, *command], check=True)
plan = json.loads((RUN / "plan.json").read_text())
print("Durasi sumber:", plan["duration"], "detik")
print("Jumlah chunk:", len(plan["jobs"]))
print("Status:", plan["status"])
print("Posisi chunk (detik):", [job["start_frame"] / 32000 for job in plan["jobs"]])
print("Konteks:", plan["chunk_seconds"], "detik; overlap nominal:", plan["overlap_seconds"], "detik")


## 3. Jalankan atau lanjutkan generasi

Chunk selesai disimpan dengan checksum. Jika proses terputus, jalankan notebook
lagi dengan sumber/profil sama: chunk selesai dipakai kembali, chunk gagal memakai
folder attempt baru. Jangan jalankan dua proses untuk folder run sama; lock OS
menolak generasi bersamaan. Mengubah seed/instrument membuat folder run baru.

Waktu tiap chunk ditampilkan. Model hanya dimuat bila masih ada chunk tersisa.
Jika semua chunk telah selesai, notebook langsung memverifikasi hasil dan
menyambungkannya. Tidak ada unduhan model pada langkah ini. Jika runtime/aset
belum siap, sel akan gagal dengan error; selesaikan notebook 02 terlebih dahulu.


In [ ]:
if DRY_RUN:
    print("DRY RUN: belum menjalankan model atau membuat stem pengiring")
else:
    assert runtime_python().is_file(), "Selesaikan instalasi notebook 02 terlebih dahulu"
    print(json.dumps(check_model_runtime(runtime_python()), indent=2))
    run_logged([str(runtime_python()), *command, "--execute"], check=True)


## 4. Dengarkan dan ekspor hasil lagu penuh

`bass_full.wav` adalah stem pengiring; `mix_full.wav` mencampurnya dengan sumber
bersih. Durasi output sama dengan hasil decode rekaman sumber, dengan sample rate
32 kHz stereo PCM16. Mix memakai gain 0.5 untuk sumber dan stem, tanpa normalisasi
berbeda per potongan. Periksa sambungan pada interval 27–30, 54–57, 81–84 detik dan seterusnya;
potongan akhir dapat memiliki overlap lebih panjang. Dengarkan juga interior setiap chunk.
Hasil tersimpan di RUN dan bisa diunduh lewat file browser JupyterLab.


In [ ]:
report_file = RUN / "experiment.json"
if report_file.exists() and json.loads(report_file.read_text()).get("status") == "generated":
    for name in (PROFILE["instrument"] + "_full.wav", "mix_full.wav"):
        path = RUN / name
        print(name, path)
        display(Audio(filename=str(path)))
else:
    print("Belum ada hasil lagu penuh yang selesai; lihat plan/status chunk di", RUN)
